# 📄 Notebook 2: Document Parsing
### How AI reads your MBA files

**WAT Reference:** `workflows/02_parse_documents.md` | `tools/parse_documents.py`

---

## What is Parsing?

Your MBA files are stored as PDFs, Word docs, PowerPoints, and Excel files. These are **binary formats** — a PDF isn't just text, it's a complex encoded file with layout information, fonts, images, metadata, and actual text all mixed together.

**Parsing** = extracting just the readable text, discarding everything else.

```
Porter_Five_Forces.pdf
(binary data: fonts, layout, images, metadata...)
         ↓  parsing
"Porter's Five Forces is a framework for analyzing
 competitive dynamics. The five forces are:
 1. Threat of new entrants..."
         ↓  ready for chunking & embedding
```

## What You'll Do in This Notebook

1. Install file-reading libraries
2. Parse a PDF — see exactly what text is extracted
3. Parse a DOCX — see how paragraphs are read
4. Parse a PPTX — see how slide text is extracted
5. Parse an XLSX — see how cell values become text
6. Auto-tag files with their subject folder
7. Parse all sample files → save to `.tmp/parsed_docs.json`

**Time needed:** ~20 minutes | **Cost:** $0 (no API calls)

---
## Step 1: Install Libraries

Each file format needs a different library:

| Library | Reads |
|---------|-------|
| `pypdf` | PDF files |
| `python-docx` | Word documents (.docx) |
| `python-pptx` | PowerPoint files (.pptx) |
| `openpyxl` | Excel files (.xlsx) |

▶ Press play — takes about 30 seconds.

In [ ]:
#@title ⚙️ Step 1 — Install file-reading libraries  { display-mode: "form" }
!pip install pypdf python-docx python-pptx openpyxl -q
print("✅ All file-reading libraries installed!")

---
## Step 2: Connect to Google Drive

Your MBA files live in Google Drive — this cell mounts your Drive so Python can access them.

> **Running in the browser (via the website)?**
> This step requires Google Drive, which only works in Google Colab.
> You can still read through the rest of this notebook to understand the parsing logic,
> but to run it with your own files, please open it in Google Colab.
> [Open in Colab →](https://colab.research.google.com/github/buildwithanshuk/MBA_RAG_Website/blob/main/notebooks/02_parse_documents.ipynb)

▶ In Colab: press play → click the link → sign in → copy the code back here.

In [ ]:
#@title 📁 Step 2 — Connect to Google Drive  { display-mode: "form" }
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except Exception:
    IN_COLAB = False
    print("ℹ️  Google Drive is not available in this environment.")
    print("   To run this notebook with your MBA files, open it in Google Colab.")

PROJECT_ROOT = '/content/drive/MyDrive/MBA_RAG'
DATA_FOLDER  = os.path.join(PROJECT_ROOT, 'data/sample')
TMP_FOLDER   = os.path.join(PROJECT_ROOT, '.tmp')

if IN_COLAB:
    os.makedirs(TMP_FOLDER, exist_ok=True)
    print(f"\n✅ Project root: {PROJECT_ROOT}")
    if os.path.exists(DATA_FOLDER):
        print("\nFiles in data/sample/:")
        for root, dirs, files in os.walk(DATA_FOLDER):
            level = root.replace(DATA_FOLDER, '').count(os.sep)
            indent = '  ' * level
            folder_name = os.path.basename(root)
            if level > 0:
                print(f"{indent}📁 {folder_name}/")
            for f in files:
                print(f"{indent}  📄 {f}")
    else:
        print(f"⚠️  data/sample/ not found. Upload your MBA sample files to: {DATA_FOLDER}")

---
## Step 3: Parse a PDF

PDF stands for Portable Document Format. A PDF file contains:
- Text (what we want)
- Fonts and layout instructions
- Images (we skip these for now)
- Page metadata

We use `pypdf` to extract just the text, page by page.

In [ ]:
#@title 📄 Step 3 — Parse a PDF  { display-mode: "form" }
import pypdf

def parse_pdf(filepath):
    text_parts = []
    reader = pypdf.PdfReader(filepath)
    total_pages = len(reader.pages)
    print(f"  PDF has {total_pages} page(s)")
    for page_num, page in enumerate(reader.pages, 1):
        page_text = page.extract_text()
        if page_text and page_text.strip():
            text_parts.append(page_text)
            print(f"  Page {page_num}: extracted {len(page_text)} characters")
        else:
            print(f"  Page {page_num}: ⚠️  no text (image-only or scanned page)")
    return "\n".join(text_parts)

pdf_file = None
for root, dirs, files in os.walk(DATA_FOLDER):
    for f in files:
        if f.lower().endswith('.pdf'):
            pdf_file = os.path.join(root, f)
            break
    if pdf_file: break

if pdf_file:
    print(f"Parsing: {os.path.basename(pdf_file)}")
    print()
    pdf_text = parse_pdf(pdf_file)
    print(f"\nTotal text extracted: {len(pdf_text):,} characters")
    print()
    print("First 500 characters:")
    print("─" * 60)
    print(pdf_text[:500])
    print("─" * 60)
else:
    print("⚠️  No PDF found in data/sample/. Upload a PDF to test this cell.")

---
## Step 4: Parse a Word Document (.docx)

Word documents store text in **paragraphs** and **tables**. The `python-docx` library gives us direct access to these structures — much cleaner than PDFs.

In [ ]:
#@title 📝 Step 4 — Parse a Word Document (.docx)  { display-mode: "form" }
import docx

def parse_docx(filepath):
    text_parts = []
    doc = docx.Document(filepath)
    para_count = 0
    for para in doc.paragraphs:
        if para.text.strip():
            text_parts.append(para.text)
            para_count += 1
    table_count = 0
    for table in doc.tables:
        for row in table.rows:
            row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
            if row_text:
                text_parts.append(row_text)
                table_count += 1
    print(f"  Paragraphs: {para_count} | Table rows: {table_count}")
    return "\n".join(text_parts)

docx_file = None
for root, dirs, files in os.walk(DATA_FOLDER):
    for f in files:
        if f.lower().endswith('.docx'):
            docx_file = os.path.join(root, f)
            break
    if docx_file: break

if docx_file:
    print(f"Parsing: {os.path.basename(docx_file)}")
    docx_text = parse_docx(docx_file)
    print(f"Total text extracted: {len(docx_text):,} characters")
    print("\nFirst 500 characters:")
    print("─" * 60)
    print(docx_text[:500])
    print("─" * 60)
else:
    print("⚠️  No .docx found in data/sample/. Upload a Word doc to test this cell.")

---
## Step 5: Parse a PowerPoint (.pptx)

PowerPoints are mostly visual — images, charts, diagrams. Only the **text in shapes** (text boxes, bullet points, titles) can be extracted.

Images inside slides are skipped (we'd need a separate OCR system for those).

In [ ]:
#@title 📊 Step 5 — Parse a PowerPoint (.pptx)  { display-mode: "form" }
from pptx import Presentation

def parse_pptx(filepath):
    text_parts = []
    prs = Presentation(filepath)
    print(f"  Presentation has {len(prs.slides)} slide(s)")
    for slide_num, slide in enumerate(prs.slides, 1):
        slide_texts = [shape.text.strip() for shape in slide.shapes if hasattr(shape, "text") and shape.text.strip()]
        if slide_texts:
            text_parts.append(f"[Slide {slide_num}] " + " ".join(slide_texts))
            print(f"  Slide {slide_num}: {len(' '.join(slide_texts))} chars")
        else:
            print(f"  Slide {slide_num}: (image-only, no text extracted)")
    return "\n".join(text_parts)

pptx_file = None
for root, dirs, files in os.walk(DATA_FOLDER):
    for f in files:
        if f.lower().endswith('.pptx'):
            pptx_file = os.path.join(root, f)
            break
    if pptx_file: break

if pptx_file:
    print(f"Parsing: {os.path.basename(pptx_file)}")
    pptx_text = parse_pptx(pptx_file)
    print(f"\nTotal text extracted: {len(pptx_text):,} characters")
    print("\nExtracted text:")
    print("─" * 60)
    print(pptx_text[:800])
    print("─" * 60)
else:
    print("⚠️  No .pptx found in data/sample/. Upload a PowerPoint to test this cell.")

---
## Step 6: Parse an Excel File (.xlsx)

Excel files are perfect for financial models, but they're mostly numbers and formulas. We extract cell values as text — formulas are replaced by their computed results (`data_only=True`).

In [ ]:
#@title 📈 Step 6 — Parse an Excel file (.xlsx)  { display-mode: "form" }
import openpyxl

def parse_xlsx(filepath):
    text_parts = []
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    print(f"  Sheets: {wb.sheetnames}")
    for sheet in wb.worksheets:
        text_parts.append(f"[Sheet: {sheet.title}]")
        row_count = 0
        for row in sheet.iter_rows(values_only=True):
            row_text = " | ".join(str(cell) for cell in row if cell is not None and str(cell).strip() not in ('', 'None'))
            if row_text:
                text_parts.append(row_text)
                row_count += 1
        print(f"  Sheet '{sheet.title}': {row_count} rows with data")
    return "\n".join(text_parts)

xlsx_file = None
for root, dirs, files in os.walk(DATA_FOLDER):
    for f in files:
        if f.lower().endswith('.xlsx'):
            xlsx_file = os.path.join(root, f)
            break
    if xlsx_file: break

if xlsx_file:
    print(f"Parsing: {os.path.basename(xlsx_file)}")
    xlsx_text = parse_xlsx(xlsx_file)
    print(f"\nTotal text extracted: {len(xlsx_text):,} characters")
    print("\nFirst 600 characters (cells joined with ' | '):")
    print("─" * 60)
    print(xlsx_text[:600])
    print("─" * 60)
else:
    print("⚠️  No .xlsx found in data/sample/. Upload an Excel file to test this cell.")

---
## Step 7: Auto-Tag Files with Subject Folders

Your 30 subject folders (Finance, Strategy, Marketing...) are a goldmine of metadata. Instead of tagging each file manually, we read the folder name automatically.

```
data/sample/Finance/CAPM.pdf          → subject: "Finance"
data/sample/Strategy/Porter.pptx      → subject: "Strategy"
data/sample/Marketing/Kotler.docx     → subject: "Marketing"
data/sample/CAPM.pdf  (no subfolder)  → subject: "General"
```

This tag travels with every chunk and enables **metadata filtering** — when you ask a Finance question, you can search only Finance chunks.

In [ ]:
#@title 🏷️ Step 7 — Auto-tag files with subject folders  { display-mode: "form" }
def get_subject(filepath, base_folder):
    rel_path = os.path.relpath(filepath, base_folder)
    parts = rel_path.split(os.sep)
    return parts[0] if len(parts) > 1 else 'General'

print("Subject auto-detection for your sample files:")
print("─" * 60)
print(f"{'Filename':<40} {'Subject':<20}")
print("─" * 60)

found_files = []
for root, dirs, files in os.walk(DATA_FOLDER):
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        if ext in ('.pdf', '.docx', '.pptx', '.xlsx'):
            filepath = os.path.join(root, f)
            subject = get_subject(filepath, DATA_FOLDER)
            print(f"{f:<40} {subject:<20}")
            found_files.append(filepath)

print("─" * 60)
print(f"Total: {len(found_files)} files")
print()
print("💡 Subject comes from the immediate parent folder name.")
print("   Files directly in data/sample/ (no subfolder) get subject = 'General'.")

---
## Step 8: Parse ALL Sample Files → Save to `.tmp/`

Now we combine everything: walk all subfolders, parse each file based on its type, tag with subject, save to `.tmp/parsed_docs.json`.

This is exactly what `tools/parse_documents.py` does.

In [ ]:
#@title 🔄 Step 8 — Parse ALL sample files  { display-mode: "form" }
import json

SUPPORTED_EXTENSIONS = {'.pdf', '.docx', '.pptx', '.xlsx'}

def parse_file(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    try:
        if ext == '.pdf':  return parse_pdf(filepath)
        if ext == '.docx': return parse_docx(filepath)
        if ext == '.pptx': return parse_pptx(filepath)
        if ext == '.xlsx': return parse_xlsx(filepath)
    except Exception as e:
        print(f"    ❌ Error: {e}")
        return ""
    return ""

documents = []
skipped   = []

all_files = []
for root, dirs, files in os.walk(DATA_FOLDER):
    for f in files:
        if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS:
            all_files.append(os.path.join(root, f))

print(f"Found {len(all_files)} files to parse")
print()

for i, filepath in enumerate(all_files, 1):
    filename = os.path.basename(filepath)
    subject  = get_subject(filepath, DATA_FOLDER)
    print(f"[{i}/{len(all_files)}] {filename}  (Subject: {subject})")
    text = parse_file(filepath)
    if not text.strip():
        print(f"  ⚠️  No text extracted — skipping")
        skipped.append(filename)
        continue
    documents.append({'filename': filename, 'source': filepath, 'subject': subject, 'text': text.strip()})
    print(f"  ✅ {len(text):,} characters extracted")

print()
print("=" * 60)
print(f"Parsed:  {len(documents)} documents")
print(f"Skipped: {len(skipped)} files (no text)")

In [ ]:
#@title 💾 Step 9 — Save results to Google Drive  { display-mode: "form" }
OUTPUT_FILE = os.path.join(TMP_FOLDER, 'parsed_docs.json')

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"✅ Saved to: {OUTPUT_FILE}")
print()
if documents:
    sample = documents[0]
    print("Structure of one entry:")
    print("─" * 60)
    print(f"  filename: {sample['filename']}")
    print(f"  subject:  {sample['subject']}")
    print(f"  text:     {sample['text'][:120]}...")
    print("─" * 60)
print(f"\nThis file has {len(documents)} documents — ready for Notebook 3.")

---
## Step 9: What Was Captured vs. Lost?

Parsing is imperfect. Let's see what we captured and what we missed.

In [ ]:
#@title 📊 Step 10 — Parsing summary  { display-mode: "form" }
from collections import Counter

print("PARSING SUMMARY")
print("=" * 60)

subjects = Counter(doc['subject'] for doc in documents)
print("\nDocuments by subject:")
for subject, count in sorted(subjects.items(), key=lambda x: -x[1]):
    total_chars = sum(len(d['text']) for d in documents if d['subject'] == subject)
    print(f"  {subject:<25} {count:>3} files   {total_chars:>10,} chars")

extensions = Counter(os.path.splitext(d['filename'])[1].lower() for d in documents)
print("\nDocuments by type:")
for ext, count in sorted(extensions.items(), key=lambda x: -x[1]):
    print(f"  {ext:<10} {count:>3} files")

print()
print("✅ CAPTURED:  Text in PDFs, Word paragraphs, PowerPoint text boxes, Excel cells")
print("⚠️  SKIPPED:  Images, charts, scanned PDFs (no text layer)")

---
## ✅ Notebook 2 Complete!

### What you learned:
1. **Parsing** = extracting text from binary file formats (PDF/DOCX/PPTX/XLSX)
2. Each file type needs a different library — each has a different internal structure
3. Your **30 subject folders become automatic tags** — no manual work
4. Images are skipped (text-only extraction in Phase 1)
5. Output saved to `.tmp/parsed_docs.json` — the input for Notebook 3

### The data flow so far:
```
data/sample/Finance/CAPM.pdf
data/sample/Strategy/Porter.pptx        → .tmp/parsed_docs.json
data/sample/Marketing/Kotler.docx
     ↑ binary files                          ↑ clean text + metadata
```

---
### Next: Notebook 3 — Text Chunking

Now that we have clean text, we need to split it into 500-token pieces. Why?
- GPT-4 can't read 100 pages at once — there's a limit
- Smaller pieces = more precise retrieval
- Each piece needs to be small enough to embed efficiently

**Before starting Notebook 3:**
1. Confirm `.tmp/parsed_docs.json` was created in your Google Drive
2. Open it and look at the structure — you should see your file names and extracted text
3. Update README.md — check off Notebook 2 ✅